# Colab Virtual VRAM 测试（简化版）

不需要 SSH，直接在 Colab 中运行

In [ ]:
# 1. 检查 GPU
!nvidia-smi

In [ ]:
# 2. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 上传代码到 Google Drive
# 然后复制到 Colab

!mkdir -p /content/APT-Transformer

# 方式 1：从 Google Drive 复制（请先手动上传）
!cp -r /content/drive/MyDrive/APT-Transformer/* /content/APT-Transformer/ 2>/dev/null || echo '请先上传代码到 Google Drive'

# 方式 2：直接在这里创建文件（用于快速测试修复）
import os
os.chdir('/content/APT-Transformer')

!echo '代码目录已准备'

In [ ]:
# 4. 安装依赖
!pip install torch datasets transformers accelerate -q
!pip install huggingface_hub -q
!echo '依赖安装完成'

In [ ]:
# 5. 修复代码：禁用错误的量化
# 直接修改 Python 文件

fix_code = '''
# 在 virtual_vram.py line 834-836 附近，找到：
# should_quantize = (LECaC_AVAILABLE and
#                    cfg.nested_quantization_bits > 0 and
#                    tensor_size_mb >= 5.0)

# 修改为（添加 enable_nested_v16 检查）：
should_quantize = (cfg.enable_nested_v16 and  # 添加这个检查
#                   LECaC_AVAILABLE and
#                   cfg.nested_quantization_bits > 0 and
#                   tensor_size_mb >= 5.0)
'''

print('修复方案：')
print(fix_code)
print('\n请在下一个单元格中应用修复')

In [ ]:
# 6. 应用修复（使用 sed）
!cd /content/APT-Transformer && sed -i 's/should_quantize = (LECaC_AVAILABLE/should_quantize = (cfg.enable_nested_v16 and \n                   LECaC_AVAILABLE/' apt/vgpu/runtime/virtual_vram.py

# 验证修改
!cd /content/APT-Transformer && grep -A3 'should_quantize' apt/vgpu/runtime/virtual_vram.py | head -10

In [ ]:
# 7. 运行测试
!cd /content/APT-Transformer && python -m apt.trainops.scripts.pretrain_quickcook \
    --output-dir ./test_vvram_lecac \
    --max-steps 50 \
    --batch-size 2 \
    --gradient-accumulation 2 \
    --use-virtual-vram \
    --vram-enable-nested-v16 \
    --vram-verbose

In [ ]:
# 8. 查看结果
# 训练完成后检查输出

!ls -la /content/APT-Transformer/test_vvram_lecac/
!echo '\n检查日志...'
!tail -50 /content/APT-Transformer/test_vvram_lecac/*.log 2>/dev/null || echo '没有找到日志文件'